In [1]:
import numpy as np
import pandas as pd

from datetime import datetime
from scipy.stats import skew 
from scipy.special import boxcox1p
from scipy.stats import boxcox_normmax
from sklearn.linear_model import ElasticNetCV, LassoCV, RidgeCV
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import mean_squared_error
from mlxtend.regressor import StackingCVRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import matplotlib.pyplot as plt
import scipy.stats as stats
import sklearn.linear_model as linear_model
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.neighbors import KDTree
from sklearn.decomposition import TruncatedSVD
from sklearn.neighbors import BallTree
from sklearn.model_selection import train_test_split
import lightgbm as lgb
import os
print(os.listdir())
import category_encoders as ce
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier
from tqdm import tqdm

import warnings
warnings.filterwarnings('ignore')

['house_price_competition_day_2_failed_trying.ipynb', 'house_price_advanced_view.ipynb', 'addr_kmeans.pkl', 'submission.csv', 'house_price_advanced_models.ipynb', 'my_model_submission.csv1', 'submission5.csv', 'my_model_submission4.csv', 'house_price', 'addr_umap.pkl', 'Day1.ipynb', 'day12.ipynb', 'titanic', 'house-prices-advanced-regression-techniques.zip', 'titanic.zip', 'my_model_submission3.csv', 'house_price_competition_view_day2.ipynb', 'Untitled.ipynb', 'my_model_submission0.csv', 'X_umap.npy', 'RD_competition_day10.ipynb', 'addr_tfidf.pkl', 'Day2 Housing Price.ipynb', 'pca_model.pkl', 'my_model_submission.csv', 'predictions.csv', 'competition_day11.ipynb', 'umap_model.pkl', 'predictions4.csv', 'getting_better_with_less_features_day_13.ipynb', 'my_model_submission1.csv', 'submission6.csv', 'Untitled1.ipynb', 'house_price_view.ipynb', 'home-data-for-ml-course.zip', '.ipynb_checkpoints', 'home-data-for-ml-course', 'house_price.ipynb', 'predictions3.csv', 'failed_trying_with_nns_da

In [9]:
# To get square foot price of neighbourhood without leaking in train
def retrieve_neighbours(model, X, y, k=5, exclude_0=False):
    # For leak-free retrival of distances and prices
    # exclude_0 = True excludes the closest neighbour (typically self when train)
    X = np.array(X)
    y = np.array(y)

    if exclude_0:
        distances, indices = model.kneighbors(X, n_neighbors=k+1)
    else:
        distances, indices = model.kneighbors(X, n_neighbors=k)

    preds = []
    dists = []
    
    for d, idxs in tqdm(zip(distances, indices), total=len(indices)):

        if exclude_0:
            d = d[1:]
            idxs = idxs[1:]
        pred = np.mean(y[idxs])
        dist = np.mean(d)
    
        preds.append(pred)
        dists.append(dist)
    
    return np.array(preds), np.array(dists)

def preprocess_knn_features(X_tr, X_va, y_tr, knn_features=["latitude","longitude","sale_year"], knn_params={'n_neighbors': 10}):
    # Features based on direct neighbourhood
    scaler = StandardScaler()
    X_tr_knn = scaler.fit_transform(X_tr[knn_features])
    X_va_knn = scaler.transform(X_va[knn_features])
    knn = KNeighborsRegressor(**knn_params).fit(X_tr_knn, y_tr)

    k = knn_params["n_neighbors"]
    
    price_tr, d_tr = retrieve_neighbours(knn, X_tr_knn, y_tr, k=k, exclude_0=True)
    price_va, d_va = retrieve_neighbours(knn, X_va_knn, y_tr, k=k, exclude_0=False)

    X_tr = X_tr.copy()
    X_va = X_va.copy()
    X_tr["k_dist"], X_va["k_dist"] = d_tr, d_va
    X_tr["price_knn"], X_va["price_knn"] = price_tr, price_va

    return X_tr, X_va

train = pd.read_csv('house_price/dataset.csv')
test = pd.read_csv('house_price/test.csv')
print ("Data is loaded!")

quantitative = [f for f in train.columns if train.dtypes[f] != 'object']
quantitative.remove('sale_price')
quantitative.remove('id')
qualitative = [f for f in train.columns if train.dtypes[f] == 'object']

sns.set_style("whitegrid")
missing = train.isnull().sum()
missing = missing[missing > 0]
print(missing)

sns.set_style("whitegrid")
missing = test.isnull().sum()
missing = missing[missing > 0]
print(missing)

def dataset_fill_null(obj):
    obj['subdivision'].fillna('Unknown', inplace=True)
    obj['sale_nbr'].fillna('Unknown', inplace=True)
    #obj.drop(columns=['sale_nbr'], inplace=True)
    obj['submarket'].fillna('Unknown', inplace=True)


dataset_fill_null(train)
dataset_fill_null(test)
print(train.shape)
print(test.shape)

# 构造原始地址字段
train_ID = train['id']
test_ID = test['id']
# Now drop the  'Id' colum since it's unnecessary for  the prediction process.
drop_cols=['id',#row_id,没有任何信息.
           'golf',#20万数据 198756都是0,基本没什么信息了.
           'view_rainier',#20万数据,198588都是0.
           'view_skyline',#20万数据,198517都是0.
           'view_lakesamm',#20万数据,198776都是0.
           'view_otherwater',#20万数据,198473都是0.
           'view_other',#20万数据,198833都是0.
          ]
train.drop(drop_cols, axis=1, inplace=True)
test_raw = test.drop(drop_cols, axis=1, inplace=True)

# Deleting outliers
train.reset_index(drop=True, inplace=True)
# We use the numpy fuction log1p which  applies log(1+x) to all elements of the column
# train["sale_price"] = np.log1p(train["sale_price"])
y = train.sale_price.reset_index(drop=True)

def preprocess_and_encode(df, y=None, encoder_bundle=None, drop_high_card=True):
    df = df.copy()
    df['total_baths'] = df['bath_full'] + 0.75*df['bath_3qtr'] + 0.5*df['bath_half']
    df['total_value'] = df['land_val'] + df['imp_val']
    df['living_area'] = df['sqft'] + df['sqft_fbsmt']

    # 补充缺失
    cat_cols = df.select_dtypes(include=['object', 'category', 'string']).columns
    df[cat_cols] = df[cat_cols].fillna('None')
    num_cols = df.select_dtypes(include=[np.number]).columns
    df[num_cols] = df[num_cols].fillna(0)

    # 日期和派生
    if 'sale_date' in df.columns:
        df['sale_date'] = pd.to_datetime(df['sale_date'])
        df['sale_year'] = df['sale_date'].dt.year
        df['sale_month'] = df['sale_date'].dt.month
        df['house_age'] = df['sale_year'] - df['year_built']
        df['reno_age'] = df['sale_year'] - df['year_reno']
        df['has_reno'] = (df['year_reno'] > 0).astype(int)
        df['land_imp_ratio'] = df['land_val'] / (df['imp_val'] + 1e-5)

    # 编码分类变量
    # 找出 object 类型列（可参与类别编码）
    cat_cols = df.select_dtypes(include=['object', 'category', 'string']).columns.tolist()
    
    # 手动优先考虑的高基类列（确保存在才用）
    manual_high_card = ['sale_date','join_status', 'city', 'zoning','subdivision','submarket']
    high_card_cols = [col for col in manual_high_card if col in df.columns and df[col].dtype == 'object']
    
    # 自动补充高基类列（nunique > 50）
    for col in cat_cols:
        if col not in high_card_cols:
            try:
                n_unique = df[col].nunique()
                if n_unique > 50:
                    high_card_cols.append(col)
            except Exception as e:
                print(f"[异常] {col}: {e}")
    
    # 低基类列自动判断（nunique <= 50）
    low_card_cols = [col for col in cat_cols if col not in high_card_cols]

    print("low",low_card_cols, "high",high_card_cols)

    if encoder_bundle is None:
        target_encoder = ce.TargetEncoder()
        X_target = target_encoder.fit_transform(df[high_card_cols], y) if y is not None else pd.DataFrame(index=df.index)
    else:
        target_encoder = encoder_bundle['target_encoder']
        X_target = target_encoder.transform(df[high_card_cols]) if high_card_cols else pd.DataFrame(index=df.index)
    X_target.columns = [f"{col}_te" for col in high_card_cols]

    # 数值标准化
    num_cols = df.select_dtypes(include=[np.number]).columns
    if encoder_bundle is None:
        scaler = StandardScaler()
        X_num_scaled = pd.DataFrame(scaler.fit_transform(df[num_cols]), columns=num_cols, index=df.index)
    else:
        scaler = encoder_bundle['scaler']
        X_num_scaled = pd.DataFrame(scaler.transform(df[num_cols]), columns=num_cols, index=df.index)

    X_final_df = pd.concat([X_num_scaled, X_target], axis=1)

    if encoder_bundle is None:
        encoder_bundle = {
            'target_encoder': target_encoder,
            'scaler': scaler
        }

    return X_final_df, encoder_bundle

# 应用预处理
# 对训练集（自己做自己）

X_train_raw, X_val, y_train, y_val = train_test_split(
    train, y, test_size=0.2, random_state=42
)

test_raw = test.copy()  # 或者正确读取原始测试集
X_train = X_train_raw.copy()
X_train = X_train.drop(['sale_price'], axis=1)
X_val = X_val.drop(['sale_price'], axis=1)
X_train_full = train.drop(['sale_price'], axis=1)
X_train, encoder_bundle = preprocess_and_encode(X_train, y_train)
X_val, _ = preprocess_and_encode(X_val, encoder_bundle=encoder_bundle)
test, _ = preprocess_and_encode(test, encoder_bundle=encoder_bundle)
X_train_full, _ = preprocess_and_encode(X_train_full, y)
X_train, X_val = preprocess_knn_features(X_train, X_val, y_train)
print("Finished")

Data is loaded!
sale_nbr       42182
subdivision    17550
submarket       1717
dtype: int64
sale_nbr       42412
subdivision    17550
submarket       1718
dtype: int64
(200000, 47)
(200000, 46)
low ['sale_nbr'] high ['join_status', 'city', 'zoning', 'subdivision', 'submarket', 'sale_warning']
low ['sale_nbr'] high ['join_status', 'city', 'zoning', 'subdivision', 'submarket', 'sale_warning']
low ['sale_nbr'] high ['join_status', 'city', 'zoning', 'subdivision', 'submarket', 'sale_warning']
low ['sale_nbr'] high ['join_status', 'city', 'zoning', 'subdivision', 'submarket', 'sale_warning']


100%|██████████████████████████████████| 40000/40000 [00:00<00:00, 95356.57it/s]

Finished


In [38]:
import optuna
from sklearn.model_selection import train_test_split
import lightgbm as lgb
from sklearn.metrics import mean_pinball_loss
import numpy as np

def winkler_score(y_true, lower, upper, alpha=0.1, return_coverage=False):
    y_true = np.asarray(y_true)
    lower = np.asarray(lower)
    upper = np.asarray(upper)

    width = upper - lower
    penalty_lower = 2 / alpha * (lower - y_true)
    penalty_upper = 2 / alpha * (y_true - upper)

    score = width.copy()
    score += np.where(y_true < lower, penalty_lower, 0)
    score += np.where(y_true > upper, penalty_upper, 0)

    if return_coverage:
        inside = (y_true >= lower) & (y_true <= upper)
        coverage = np.mean(inside)
        return np.mean(score), coverage

    return np.mean(score)

# ========= 数据划分 =========
X_tr, X_val_, y_tr, y_val_ = X_train, X_val, y_train, y_val

def train_quantile_model_lgb(X, y, alpha, params):
    params = params.copy()
    params.update({
        'objective': 'quantile',
        'alpha': alpha,
        'verbosity': -1
    })
    model = lgb.LGBMRegressor(**params)
    model.fit(X, y,
              eval_set=[(X_val_, y_val_)],
               callbacks=[lgb.early_stopping(stopping_rounds=50), lgb.log_evaluation(0)]
                )
    return model

In [6]:
def objective(trial):
    common_params = {
        'n_estimators': trial.suggest_int('n_estimators', 500, 3000, step=500),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 5, 64),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 10, 100, step=10),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.6, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.6, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),
        'lambda_l1': trial.suggest_float('lambda_l1', 0.0, 5.0),
        'lambda_l2': trial.suggest_float('lambda_l2', 0.0, 5.0),
    }

    lower_model = train_quantile_model_lgb(X_tr, y_tr, alpha=0.05, params=common_params)
    upper_model = train_quantile_model_lgb(X_tr, y_tr, alpha=0.95, params=common_params)

    pred_lower = lower_model.predict(X_val_)
    pred_upper = upper_model.predict(X_val_)

    score = winkler_score(y_val_, pred_lower, pred_upper, alpha=0.1)
    return score  # 越小越好

# ========= 启动搜索 =========
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=100, n_jobs=4)
print("Best params:", study.best_params)



[I 2025-07-15 21:45:13,427] A new study created in memory with name: no-name-726d0639-feda-4ee5-aa51-c0a126ca8efb


Training until validation scores don't improve for 50 rounds
Training until validation scores don't improve for 50 rounds
Training until validation scores don't improve for 50 rounds
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[500]	valid_0's quantile: 8144.92
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[500]	valid_0's quantile: 7833.38
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[500]	valid_0's quantile: 10251.6
Did not meet early stopping. Best iteration is:
[500]	valid_0's quantile: 7659.09


[I 2025-07-15 21:45:35,863] Trial 3 finished with value: 367930.30258851807 and parameters: {'n_estimators': 500, 'learning_rate': 0.06321813768488904, 'num_leaves': 11, 'min_data_in_leaf': 10, 'feature_fraction': 0.9439295189200386, 'bagging_fraction': 0.6435993082798521, 'bagging_freq': 2, 'lambda_l1': 1.1514680643002646, 'lambda_l2': 2.9448952405278606}. Best is trial 3 with value: 367930.30258851807.


Training until validation scores don't improve for 50 rounds
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[1000]	valid_0's quantile: 7865.9
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[500]	valid_0's quantile: 9534.79


[I 2025-07-15 21:45:54,159] Trial 2 finished with value: 347363.2243639617 and parameters: {'n_estimators': 500, 'learning_rate': 0.03886870442420458, 'num_leaves': 51, 'min_data_in_leaf': 80, 'feature_fraction': 0.768304860600176, 'bagging_fraction': 0.6679610903779071, 'bagging_freq': 5, 'lambda_l1': 2.6285806632172912, 'lambda_l2': 2.0697146695124227}. Best is trial 2 with value: 347363.2243639617.


...

Did not meet early stopping. Best iteration is:
[2999]	valid_0's quantile: 9131.14


[I 2025-07-15 22:41:06,914] Trial 98 finished with value: 331756.63782636577 and parameters: {'n_estimators': 3000, 'learning_rate': 0.03566744958433199, 'num_leaves': 18, 'min_data_in_leaf': 80, 'feature_fraction': 0.6840303968220959, 'bagging_fraction': 0.7913527277746718, 'bagging_freq': 3, 'lambda_l1': 2.6564757001412405, 'lambda_l2': 3.622079746393673}. Best is trial 84 with value: 329965.97540820163.


Did not meet early stopping. Best iteration is:
[2999]	valid_0's quantile: 9113.06


[I 2025-07-15 22:41:14,580] Trial 99 finished with value: 332177.51491489564 and parameters: {'n_estimators': 3000, 'learning_rate': 0.027399935646895815, 'num_leaves': 20, 'min_data_in_leaf': 80, 'feature_fraction': 0.7641727678269884, 'bagging_fraction': 0.7576215375816436, 'bagging_freq': 3, 'lambda_l1': 1.6633339401932756, 'lambda_l2': 2.1295934675872106}. Best is trial 84 with value: 329965.97540820163.


Best params: {'n_estimators': 3000, 'learning_rate': 0.04636037450615781, 'num_leaves': 19, 'min_data_in_leaf': 100, 'feature_fraction': 0.7298036912145577, 'bagging_fraction': 0.8129812092810939, 'bagging_freq': 4, 'lambda_l1': 1.173467234602045, 'lambda_l2': 2.576496765290143}


In [40]:
def winkler_score(y_true, lower, upper, alpha=0.1, return_coverage=False):
    y_true = np.asarray(y_true)
    lower = np.asarray(lower)
    upper = np.asarray(upper)

    width = upper - lower
    penalty_lower = 2 / alpha * (lower - y_true)
    penalty_upper = 2 / alpha * (y_true - upper)

    score = width.copy()
    score += np.where(y_true < lower, penalty_lower, 0)
    score += np.where(y_true > upper, penalty_upper, 0)

    if return_coverage:
        inside = (y_true >= lower) & (y_true <= upper)
        coverage = np.mean(inside)
        return np.mean(score), coverage

    return np.mean(score)
    
best_parmas = {'n_estimators': 3000, 'learning_rate': 0.04636037450615781, 'num_leaves': 19, 'min_data_in_leaf': 100, 'feature_fraction': 0.7298036912145577, 'bagging_fraction': 0.8129812092810939, 'bagging_freq': 4, 'lambda_l1': 1.173467234602045, 'lambda_l2': 2.576496765290143}
lower_model_lgb = train_quantile_model_lgb(X_train, y_train, alpha=0.05, params=best_parmas)
upper_model_lgb = train_quantile_model_lgb(X_train, y_train, alpha=0.95, params=best_parmas)

pred_lower = lower_model_lgb.predict(X_val)
pred_upper = upper_model_lgb.predict(X_val)

score = winkler_score(y_val, pred_lower, pred_upper, alpha=0.1, return_coverage=True)
print(score)
score = winkler_score(y_val, pred_lower-10000, pred_upper+10000, alpha=0.1, return_coverage=True)
print(score)

Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[2569]	valid_0's quantile: 7424.85
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[3000]	valid_0's quantile: 9073.45
(329965.97540820163, 0.85805)
(326770.47561395366, 0.906125)


In [57]:
score = winkler_score(y_val, pred_lower-8500, pred_upper+8500, alpha=0.1, return_coverage=True)
print(score)

(326669.28165217733, 0.900275)


In [45]:
def best_features(model, num=50):
    # 获取特征重要性
    importance = model.feature_importances_
    features = X_train.columns
    
    # 打包成 DataFrame
    feat_imp = pd.DataFrame({
        'feature': features,
        'importance': importance
    }).sort_values('importance', ascending=False)
    
    # 显示前 30 个最重要的特征
    print(feat_imp.head(num))



In [59]:
def train_quantile_model_lgb_full(X, y, alpha, params):
    params = params.copy()
    params.update({
        'objective': 'quantile',
        'alpha': alpha,
        'verbosity': -1
    })
    model = lgb.LGBMRegressor(**params)
    model.fit(X, y)
    return model
    
#best_params = {'n_estimators': 3000, 'learning_rate': 0.017548411396214044, 'num_leaves': 54, 'min_data_in_leaf': 100, 'feature_fraction': 0.6454574945988764, 'bagging_fraction': 0.8467545991300843, 'bagging_freq': 3, 'lambda_l1': 4.291976770761236, 'lambda_l2': 3.428080554361697}
best_parmas = {'n_estimators': 3000, 'learning_rate': 0.04636037450615781, 'num_leaves': 19, 'min_data_in_leaf': 100, 'feature_fraction': 0.7298036912145577, 'bagging_fraction': 0.8129812092810939, 'bagging_freq': 4, 'lambda_l1': 1.173467234602045, 'lambda_l2': 2.576496765290143}
lower_model_lgb = train_quantile_model_lgb_full(X_train_full, y, alpha=0.05, params=best_params)
upper_model_lgb = train_quantile_model_lgb_full(X_train_full, y, alpha=0.95, params=best_params)

pred_lower_lgb = lower_model_lgb.predict(test)
pred_upper_lgb = upper_model_lgb.predict(test)

In [11]:
from catboost import CatBoostRegressor, Pool
import optuna

def train_cb_quantile_model(X, y, alpha, params):
    params = params.copy()
    params.update({
        'loss_function': f'Quantile:alpha={alpha}',
        'verbose': 0
    })

    train_pool = Pool(X, y)
    val_pool = Pool(X_val_, y_val_)

    model = CatBoostRegressor(**params)
    model.fit(train_pool, eval_set=val_pool, early_stopping_rounds=50)

    return model

def objective_cb(trial):
    common_params = {
        'iterations': trial.suggest_int('iterations', 500, 3000, step=500),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'depth': trial.suggest_int('depth', 4, 10),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1.0, 10.0),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0, 1.0),
        'random_strength': trial.suggest_float('random_strength', 1e-9, 10.0, log=True),
        'border_count': trial.suggest_int('border_count', 32, 255),
    }

    lower_model = train_cb_quantile_model(X_tr, y_tr, alpha=0.05, params=common_params)
    upper_model = train_cb_quantile_model(X_tr, y_tr, alpha=0.95, params=common_params)

    pred_lower = lower_model.predict(X_val_)
    pred_upper = upper_model.predict(X_val_)

    score = winkler_score(y_val_, pred_lower, pred_upper, alpha=0.1)
    return score

# 启动搜索
study = optuna.create_study(direction='minimize')
study.optimize(objective_cb, n_trials=100, n_jobs=4)

print("Best params:", study.best_params)


[I 2025-07-15 22:48:04,782] A new study created in memory with name: no-name-35c0291f-5b51-42a2-92f7-01459c0ed958
[I 2025-07-15 22:49:35,383] Trial 1 finished with value: 344551.69309429236 and parameters: {'iterations': 1000, 'learning_rate': 0.038365471213488544, 'depth': 7, 'l2_leaf_reg': 9.434755474893887, 'bagging_temperature': 0.5142398837910376, 'random_strength': 0.008055408584155791, 'border_count': 146}. Best is trial 1 with value: 344551.69309429236.
[I 2025-07-15 22:49:41,888] Trial 2 finished with value: 345608.4021648109 and parameters: {'iterations': 2000, 'learning_rate': 0.048762533809435744, 'depth': 8, 'l2_leaf_reg': 4.723064265619694, 'bagging_temperature': 0.8484528369120127, 'random_strength': 0.006375237451126787, 'border_count': 188}. Best is trial 1 with value: 344551.69309429236.
[I 2025-07-15 22:50:09,020] Trial 4 finished with value: 419609.6389685562 and parameters: {'iterations': 500, 'learning_rate': 0.021783834519061334, 'depth': 5, 'l2_leaf_reg': 8.3575

Best params: {'iterations': 3000, 'learning_rate': 0.04809304932404509, 'depth': 5, 'l2_leaf_reg': 7.430597979768241, 'bagging_temperature': 0.8625782660458902, 'random_strength': 6.857105332279787e-05, 'border_count': 119}


In [ ]:
best_params = {'iterations': 3000, 'learning_rate': 0.04809304932404509, 'depth': 5, 'l2_leaf_reg': 7.430597979768241, 'bagging_temperature': 0.8625782660458902, 'random_strength': 6.857105332279787e-05, 'border_count': 119}

lower_model = train_cb_quantile_model(X_train, y_train, alpha=0.05, params=best_params)
upper_model = train_cb_quantile_model(X_train, y_train, alpha=0.95, params=best_params)

pred_lower = lower_model.predict(X_val)
pred_upper = upper_model.predict(X_val)

score = winkler_score(y_val, pred_lower, pred_upper, alpha=0.1)
print(score)

best_features(lower_model)
best_features(upper_model)

In [12]:
#  训练模型
def train_quantile_model(alpha,mdoel_name):
    if mdoel_name=='cat':
        cat_params = {
            'objective': f'Quantile:alpha={alpha}',             # 回归任务，使用均方根误差
            'learning_rate': 0.05,           # 学习率
            'iterations': 8000,              # 树的数量
            'random_seed': 42,               # 随机种子
            'verbose': 800,                   # 显示训练过程
            'grow_policy' :"Depthwise",
            'min_data_in_leaf': 1000,
            'l2_leaf_reg': 100,
            'od_type':"IncToDec",
            'od_pval':0.1,
        }
        model = CatBoostRegressor(**cat_params)
        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
        )
        return model

cat_model_lower = train_quantile_model(0.05,"cat")
print("cat_model_lower is ok")
cat_model_upper = train_quantile_model(0.95,"cat")
print('cat_model_upper is ok')

cat_lower = cat_model_lower.predict(X_val)
cat_upper = cat_model_upper.predict(X_val)

score = winkler_score(y_val, cat_lower, cat_upper, alpha=0.1)
print(score)

best_features(cat_model_lower)
best_features(cat_model_upper)

0:	learn: 21174.0855061	test: 21125.6647813	best: 21125.6647813 (0)	total: 28.3ms	remaining: 3m 46s
800:	learn: 6944.9627326	test: 7556.4634763	best: 7556.4634763 (800)	total: 21.7s	remaining: 3m 15s
1600:	learn: 6515.0463082	test: 7390.0856143	best: 7390.0752694 (1599)	total: 43.6s	remaining: 2m 54s
2400:	learn: 6317.6151246	test: 7344.6000520	best: 7344.3830467 (2398)	total: 1m 5s	remaining: 2m 33s
3200:	learn: 6184.4063474	test: 7328.7928826	best: 7328.7928826 (3200)	total: 1m 27s	remaining: 2m 11s
4000:	learn: 6086.9772060	test: 7320.3784368	best: 7320.2897702 (3978)	total: 1m 49s	remaining: 1m 49s
Stopped by overfitting detector  (20 iterations wait)

bestTest = 7319.132149
bestIteration = 4121

Shrink model to first 4122 iterations.
cat_model_lower is ok
0:	learn: 62620.7560805	test: 62755.0781398	best: 62755.0781398 (0)	total: 25.3ms	remaining: 3m 22s
800:	learn: 8126.8469905	test: 9374.4321567	best: 9374.4321567 (800)	total: 22.7s	remaining: 3m 24s
1600:	learn: 7509.6897416	tes

NameError: name 'best_features' is not defined

In [60]:
score = winkler_score(y_val, cat_lower-8400, cat_upper+8400, alpha=0.1, return_coverage=True)
print(score)

(324494.676749847, 0.9009)


In [62]:
#  训练模型
def train_quantile_model_cat_full(alpha):
    if mdoel_name=='cat':
        cat_params = {
            'objective': f'Quantile:alpha={alpha}',             # 回归任务，使用均方根误差
            'learning_rate': 0.05,           # 学习率
            'iterations': 8000,              # 树的数量
            'random_seed': 42,               # 随机种子
            'verbose': 800,                   # 显示训练过程
            'grow_policy' :"Depthwise",
            'min_data_in_leaf': 1000,
            'l2_leaf_reg': 100,
            'od_type':"IncToDec",
            'od_pval':0.1,
        }
        model = CatBoostRegressor(**cat_params)
        model.fit(X_train_full, y)
        return model

cat_model_lower_full = train_quantile_model_cat_full(0.05)
cat_model_upper_full = train_quantile_model_cat_full(0.95)

cat_lower = cat_model_lower_full.predict(test)
cat_upper = cat_model_upper_full.predict(test)

0:	learn: 21189.2367502	total: 36ms	remaining: 4m 48s
800:	learn: 7057.9158916	total: 27.3s	remaining: 4m 5s
1600:	learn: 6683.9040820	total: 55.2s	remaining: 3m 40s
2400:	learn: 6478.4383065	total: 1m 22s	remaining: 3m 12s
3200:	learn: 6356.0029374	total: 1m 50s	remaining: 2m 45s
4000:	learn: 6267.2754357	total: 2m 17s	remaining: 2m 17s
4800:	learn: 6196.7022900	total: 2m 44s	remaining: 1m 49s
5600:	learn: 6135.9011413	total: 3m 11s	remaining: 1m 22s
6400:	learn: 6087.9216930	total: 3m 39s	remaining: 54.7s
7200:	learn: 6048.9173157	total: 4m 6s	remaining: 27.3s
7999:	learn: 6004.8716314	total: 4m 33s	remaining: 0us
cat_model_lower is ok
0:	learn: 62698.2056431	total: 31.4ms	remaining: 4m 11s
800:	learn: 8143.3422303	total: 27.2s	remaining: 4m 4s
1600:	learn: 7645.6443273	total: 54.2s	remaining: 3m 36s
2400:	learn: 7387.5971654	total: 1m 21s	remaining: 3m 10s
3200:	learn: 7229.9396693	total: 1m 48s	remaining: 2m 43s
4000:	learn: 7115.0513513	total: 2m 15s	remaining: 2m 15s
4800:	learn:

In [63]:
pre_low = (pred_lower_lgb-8500+cat_lower-8400)/2
pre_high = (pred_upper_lgb+8500+cat_upper+8400)/2
print("Finished")

Finished


In [65]:
submission = pd.DataFrame({
        "id": test_ID,
        "pi_lower": pre_low,
        "pi_upper": pre_high
    })
submission.to_csv("submission0.csv", index=False)

In [18]:
score = winkler_score(y_val, cat_lower-7500, cat_upper+7500, alpha=0.1, return_coverage=True)
print(score)

(324510.5535037445, 0.8974)


In [19]:
score = winkler_score(y_val, cat_lower-10000, cat_upper+10000, alpha=0.1, return_coverage=True)
print(score)

(324618.84047304146, 0.906875)


(324494.676749847, 0.9009)
